In [ ]:
# ===== STATIC RANK-4/5 STABILIZATION (T2) — CONFIG (edit ONLY this cell) ==
# The mentor's un-run "dumbest possible thing": adjoin 2-3 coupled relators
# z_i^-1 w_i up front and search at n_gen=4/5 with the general-n solver.
# Defaults live in experiments/stable_ac/nocov/config_static_rank.yaml;
# OVERRIDES here win. One jsonl per (budget, family, rank); resume key =
# (name, z_words) -> relaunching is always safe.

REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH   = "research/stable-ac-escape"
REPO_DIR = "ACSolverX"
UPDATE_REPO = True

MOUNT_DRIVE = True
DRIVE_DIR   = "/content/drive/MyDrive/acsolverx_results/stable_ac_static_rank"

CONFIG_PATH = "experiments/stable_ac/nocov/config_static_rank.yaml"
OVERRIDES = {
    "BUDGET": [10000],          # production ladder: 10k first, then 50k on the survivors
    "RANKS": [4, 5],
    "FAMILIES": ["A1"],
    "MOUNT_DRIVE": True,
}


In [ ]:
# ==================== SETUP (clone / pull / install) ======================
import os, sys, subprocess

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)                       # anchor so re-runs never nest the clone
    if not os.path.isdir(REPO_DIR):
        sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy pyyaml")
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
else:
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
# a RESTARTED kernel may hold stale modules from before a git reset — purge
for m in [k for k in sys.modules if k == "experiments" or k.startswith("experiments.")]:
    del sys.modules[m]
print("repo root:", REPO_ROOT)


In [ ]:
# ==================== RUN =================================================
os.environ["ACSOLVERX_ALLOW_BIG"] = "1"
import yaml
from experiments.stable_ac.nocov.run_static_rank import run_static_rank

cfg = yaml.safe_load(open(CONFIG_PATH))
cfg.update(OVERRIDES)
if IN_COLAB and MOUNT_DRIVE:
    cfg["DRIVE_OUT_DIR"] = DRIVE_DIR
for budget in cfg["BUDGET"]:
    for family in cfg["FAMILIES"]:
        for rank in cfg["RANKS"]:
            out = run_static_rank(cfg, budget, family, rank)
            print("written:", out)
# Verify every solved row before belief:
#   .venv/bin/python3 -m experiments.stable_ac.verify_static_rank results/stable_ac/static_rank
